In [87]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [88]:
#df= pd.read_csv('electronics_transactions .csv')

separators = [',', ';', '\t', '|']

for sep in separators:
    try:
        df = pd.read_csv('electronics_transactions .csv', sep = sep)
        print(sep, ' - успешниый разделитель')
        break
    except Exception as ex:
        print(sep, 'Не разделитель')

df.columns = ['1']
df = df['1'].str.split(',', expand = True)
df = df.drop(0, axis = 1)
df

, Не разделитель
;  - успешниый разделитель


,1,2,3,4,5
0,Смартфон,Чехол,Защитное_стекло,None,None
1,Ноутбук,Чехол_для_ноутбука,Мышь,None,None
2,Планшет,Стилус,Клавиатура,Чехол,None
3,Смартфон,Наушники,Чехол,Защитное_стекло,None
4,Ноутбук,Мышь,Охлаждающая_подставка,None,None
...,...,...,...,...,...
95,Планшет,Стилус,Чехол,Защитное_стекло,Клавиатура
96,Смартфон,Портативный_аккумулятор,Кабель,Автодержатель,Чехол
97,Наушники,Чехол_для_наушников,Кабель,Bluetooth_адаптер,None
98,Ноутбук,Мышь,Коврик_для_мыши,Охлаждающая_подставка,Чистящие_средства


In [89]:
df.value_counts()


1         2                        3                4                        5                      
Смартфон  Чехол                    Наушники         Защитное_стекло          Портативный_аккумулятор    2
Ноутбук   Мышь                     Веб_камера       Наушники                 Микрофон                   1
                                   Коврик_для_мыши  Охлаждающая_подставка    Чистящие_средства          1
                                   Клавиатура       Сумка_для_ноутбука       Охлаждающая_подставка      1
          Сумка_для_ноутбука       Мышь             Чехол_для_ноутбука       Охлаждающая_подставка      1
Планшет   Стилус                   Чехол            Защитное_стекло          Клавиатура                 1
Смартфон  Защитное_стекло          Чехол            Наушники                 Карта_памяти               1
Планшет   Чехол                    Стилус           Портативный_аккумулятор  Кабель                     1
Смартфон  Портативный_аккумулятор  Кабель          

In [90]:
total_counts = pd.Series(dtype=int)

for i in range(1, 6):
    total_counts = total_counts.add(df[i].value_counts(), fill_value=0)

total_counts = total_counts.astype(int).sort_values(ascending = False)

sample_sorting = total_counts.index
total_counts

Чехол                                47
Смартфон                             40
Наушники                             33
Защитное_стекло                      33
Ноутбук                              30
Мышь                                 28
Портативный_аккумулятор              16
Планшет                              15
Стилус                               14
Клавиатура                           13
Кабель                               11
Охлаждающая_подставка                11
Сумка_для_ноутбука                    9
Веб_камера                            8
Чехол_для_наушников                   6
Коврик_для_мыши                       5
Карта_памяти                          5
Игра                                  5
Умные_часы                            5
Игровая_консоль                       5
Чистящие_средства                     4
Bluetooth_адаптер                     3
Микрофон                              3
Подарочная_карта                      3
Адаптер                               3


In [91]:
df_scan = df.copy()
for i in range(len(df_scan)):
    row_serie = pd.Series(index = df_scan.iloc[i].dropna(), data = total_counts[df_scan.iloc[i].dropna()]).sort_values(ascending= False)
    df_scan.iloc[i, :len(row_serie)] = row_serie.index.tolist()
df_scan

,1,2,3,4,5
0,Чехол,Смартфон,Защитное_стекло,None,None
1,Ноутбук,Мышь,Чехол_для_ноутбука,None,None
2,Чехол,Планшет,Стилус,Клавиатура,None
3,Чехол,Смартфон,Наушники,Защитное_стекло,None
4,Ноутбук,Мышь,Охлаждающая_подставка,None,None
...,...,...,...,...,...
95,Чехол,Защитное_стекло,Планшет,Стилус,Клавиатура
96,Чехол,Смартфон,Портативный_аккумулятор,Кабель,Автодержатель
97,Наушники,Кабель,Чехол_для_наушников,Bluetooth_адаптер,None
98,Ноутбук,Мышь,Охлаждающая_подставка,Коврик_для_мыши,Чистящие_средства


In [145]:
class Bor:
    class Node:
        def __init__(self, value=None, count=0):
            self.value = value       
            self.count = count        
            self.children = {}        
            self.parent = None        
            self.link = None          # связь для header table (узел с тем же value)
        
        def __repr__(self):
            return f"Node({self.value}:{self.count})"
    
    def __init__(self):
        self.root = self.Node()       
        self.header_table = {}        # таблица заголовков {value: [Node1, Node2, ...]}
        self.item_counts = total_counts         # частоты одиночных элементов
    
    def add_transaction(self, transaction):
        current_node = self.root
        for string in transaction:
            if string not in current_node.children:
                current_node.children[string] = self.Node(string, 0)
                current_node.children[string].parent = current_node
                
                # Обновляем header table
                if string not in self.header_table:
                    self.header_table[string] = []
                self.header_table[string].append(current_node.children[string])
                
            current_node = current_node.children[string]
            current_node.count += 1
        
    def build_trie(self, transactions, min_support=0.15):
        min_count = min_support * len(transactions)
        
        # Абсолютная частота проще
        frequent_items = {item: count for item, count in self.item_counts.items() 
                         if count >= min_count}
        
        # Сортировка частых элементов
        sorted_frequent = sorted(frequent_items.items(), 
                               key=lambda x: (-x[1], x[0]))
        
        for transaction in transactions:
            filtered_trans = [item for item in transaction if item in frequent_items]
            if filtered_trans:
                # Сортируем по убыванию частоты
                self.add_transaction(filtered_trans)
            
    def fing_frequent_sets(self, min_support):
        min_support = min_support * len(df_scan)
        current_node = self.root
        results = []
        for item, nodes in self.header_table.items():
            total_count = sum([node.count for node in nodes])
            if total_count >= min_support:
                results.append([item])
                for node in nodes:
                    if node.count >= min_support:
                        itemset = [node.value]
                        parent = node.parent
                        while parent and parent.value is not None:
                            itemset.append(parent.value)
                            results.append(itemset[1:].copy())  # БЕЗ ИСХОДНОГО
                            parent = parent.parent
                            
        final_results = []
        seen = set()
        for itemset in results:
            key = tuple(sorted(itemset))
            if key not in seen:
                seen.add(key)
                final_results.append(itemset)
        
        return final_results
        
        

In [146]:
bor = Bor()
transactions = df_scan.apply(lambda row: row.dropna().tolist(), axis=1).tolist()
print(f"Всего транзакций: {len(transactions)}")

bor.build_trie(transactions, min_support=0.15)
print(f"Построено дерево, header_table: {len(bor.header_table)} элементов")

frequent_sets = bor.fing_frequent_sets(0.15)
print(f"Найдено {len(frequent_sets)} частых наборов:")
for itemset in frequent_sets[:20]:  # покажем первые 20
    print(f"  {itemset}")

Всего транзакций: 100
Построено дерево, header_table: 8 элементов
Найдено 9 частых наборов:
  ['Чехол']
  ['Смартфон']
  ['Защитное_стекло']
  ['Смартфон', 'Чехол']
  ['Ноутбук']
  ['Мышь']
  ['Планшет']
  ['Наушники']
  ['Портативный_аккумулятор']


In [147]:
frequent_sets

[['Чехол'],
 ['Смартфон'],
 ['Защитное_стекло'],
 ['Смартфон', 'Чехол'],
 ['Ноутбук'],
 ['Мышь'],
 ['Планшет'],
 ['Наушники'],
 ['Портативный_аккумулятор']]